In [64]:
import os 
import requests

pdf_path = "human-nutrition-text.pdf"

if not os.path.exists(pdf_path):
    print(f"[INFO] File doesn't exist, downloading....")

    url = "https://pressbooks.oer.hawaii.edu/humannutrition2/open/download?type=pdf"
    filename = pdf_path
    response = requests.get(url)
    #check if the request was successful
    if response.status_code == 200:
        with open(filename, "wb") as file:
            file.write(response.content)
        print(f"[INFO] the file has been downloaded and saved as {filename}")
    else:
        print(f"[INFO] Failed to download the file. status code: {response.status_code}")
else:
    print(f"File {pdf_path} already exists")

File human-nutrition-text.pdf already exists


In [65]:
import fitz
from tqdm.auto import tqdm

def text_formatter(text: str) -> str:
    """ Perfroms minor formatting on text."""
    cleaned_text = text.replace("\n"," ").strip()
    return cleaned_text

def open_and_read_pdf(pdf_path: str) -> list[dict]:
    doc = fitz.open(pdf_path)
    pages_and_texts = []
    for page_number, page in tqdm(enumerate(doc)):
        text = page.get_text()
        text = text_formatter(text=text)
        pages_and_texts.append({"page_number" : page_number - 41,
                                "page_char_count" : len(text),
                                "page_word_count" : len(text.split(" ")),
                                "page_sentence_count_raw" : len(text.split(". ")),
                                "page_token_count" : len(text) /4, # we assume that 1 token = ~4 characters
                                "text" : text   
        })
    return pages_and_texts

pages_and_texts = open_and_read_pdf(pdf_path=pdf_path)
pages_and_texts[:2]

0it [00:00, ?it/s]

[{'page_number': -41,
  'page_char_count': 29,
  'page_word_count': 4,
  'page_sentence_count_raw': 1,
  'page_token_count': 7.25,
  'text': 'Human Nutrition: 2020 Edition'},
 {'page_number': -40,
  'page_char_count': 0,
  'page_word_count': 1,
  'page_sentence_count_raw': 1,
  'page_token_count': 0.0,
  'text': ''}]

In [66]:
import random

random.sample(pages_and_texts, k=3)

[{'page_number': 90,
  'page_char_count': 852,
  'page_word_count': 157,
  'page_sentence_count_raw': 6,
  'page_token_count': 213.0,
  'text': 'Blood tests  are helpful  tools in  diagnosing  disease and  provide  much  information  on overall  health. Image  by Thirteen  of Clubs /  CC BY-SA  Since blood is the conduit of metabolic products and wastes,  measuring the components of blood, and particular substances in  blood, can reveal not only the health of blood, but also the health  of other organ systems. In standard blood tests performed during  an annual physical, the typical blood tests conducted can tell your  physician about the functioning of a particular organ or about  disease risk.  A biomarker is defined as a measurable molecule or trait that  is connected with a specific disease or health condition. The  concentrations of biomarkers in blood are indicative of disease risk.  Some biomarkers are cholesterol, triglycerides, glucose, and  90  |  The Cardiovascular System'},

In [67]:
import pandas as pd

df = pd.DataFrame(pages_and_texts)
df.head()

,page_number,page_char_count,page_word_count,page_sentence_count_raw,page_token_count,text
0,-41,29,4,1,7.25,Human Nutrition: 2020 Edition
1,-40,0,1,1,0.00,
2,-39,320,54,1,80.00,Human Nutrition: 2020 Edition UNIVERSITY OF ...
3,-38,212,32,1,53.00,Human Nutrition: 2020 Edition by University of...
4,-37,797,145,2,199.25,Contents Preface University of Hawai‘i at Mā...


In [68]:
df.describe().round(2)

,page_number,page_char_count,page_word_count,page_sentence_count_raw,page_token_count
count,1208.00,1208.00,1208.00,1208.00,1208.00
mean,562.50,1148.00,198.30,9.97,287.00
std,348.86,560.38,95.76,6.19,140.10
min,-41.00,0.00,1.00,1.00,0.00
25%,260.75,762.00,134.00,4.00,190.50
50%,562.50,1231.50,214.50,10.00,307.88
75%,864.25,1603.50,271.00,14.00,400.88
max,1166.00,2308.00,429.00,32.00,577.00


Why would we care about token count?

Token couunt is important to think about because:
1. Embedding models don't deal with infinite tokens.
2. LLMs don't deal with infinite tokens.

for example an embedding model may have been trained to embed sequences of 384 token into numerical space (sentence-transformers 'all-mpnet-base-v2')

As fro LLMs, they can't accept infinite tokens in their context window.

## Text Processing (Splitting pages into sentences)

Two ways to do this:
1. we've done that by using splitting on '". "'.
2. We can use spaCy (NLP library) 

In [69]:
from spacy.lang.en import English
nlp = English()

# Add a sentencizer piepeline
nlp.add_pipe("sentencizer")

# create document instance as an example
doc = nlp("This is a sentence. This another sentence. Ilike elephents.")
assert len(list(doc.sents)) == 3

# print out our sentences split
list(doc.sents)

[This is a sentence., This another sentence., Ilike elephents.]

In [70]:
for item in tqdm(pages_and_texts):
    item["sentences"] = list(nlp(item["text"]).sents)
    # Make sure all sentences are string default 
    item["sentences"] = [str(sentence) for sentence in item["sentences"]]
    item["page_sentence_count_spacy"] = len(item["sentences"])
    

  0%|          | 0/1208 [00:00<?, ?it/s]

In [71]:
random.sample(pages_and_texts,k=1)

[{'page_number': 498,
  'page_char_count': 749,
  'page_word_count': 118,
  'page_sentence_count_raw': 4,
  'page_token_count': 187.25,
  'text': 'US state  map of  obesity  prevalence  https://w ww.cdc.go v/obesity/ data/ prevalence -maps.ht ml  State Map of the Prevalence of Obesity  in America  \xa0 Visit https://www.cdc.gov/obesity/data/prevalence- maps.html to see the prevalence of self-reported obesity  among U.S. adults from 2014-2016.  \xa0 As BMIs increase over 25, the risks increase for heart disease,  Type 2 diabetes, hypertension, endometrial cancer, postmenopausal  breast cancer, colon cancer, stroke, osteoarthritis, liver disease,  gallbladder disorders, and hormonal disorders. The WHO reports  that overweight and obesity are the fifth leading cause for deaths  globally, and estimates that more than 2.8 million adults die annually  498  |  Factors Affecting Energy Expenditure',
  'sentences': ['US state  map of  obesity  prevalence  https://w ww.cdc.go v/obesity/ data/ pr

In [72]:
df= pd.DataFrame(pages_and_texts)
df.describe().round(2)

,page_number,page_char_count,page_word_count,page_sentence_count_raw,page_token_count,page_sentence_count_spacy
count,1208.00,1208.00,1208.00,1208.00,1208.00,1208.00
mean,562.50,1148.00,198.30,9.97,287.00,10.32
std,348.86,560.38,95.76,6.19,140.10,6.30
min,-41.00,0.00,1.00,1.00,0.00,0.00
25%,260.75,762.00,134.00,4.00,190.50,5.00
50%,562.50,1231.50,214.50,10.00,307.88,10.00
75%,864.25,1603.50,271.00,14.00,400.88,15.00
max,1166.00,2308.00,429.00,32.00,577.00,28.00


### Chunking our sentences together

The concept of splitting larger pieces of text into smaller ones it often referred to as text splitting or chunking.

There is no 100% correct way to do this.

We'll keep it simple and split into groups of 10 sentences (however, you could also try 5,7,8. whatever you like)

There aree  frameworks such as langchain which can help with this, however, we'll stick with python now.

Why we do this:
1. So our texts are easier to filter (smaller group of text can be easier to inspect that large passages of text).
2. So our text chunks can fit into our embedding model context window(eg. 384 tokens as a limit)
3. So our contexts passed to an LLM can be more specific and focused.


In [73]:
# Define split size to turn groups of sentences into chunks
num_sentence_chunk_size = 10

#create a function to split lists of texts recursively into chuncks size
# e.g, [20] -> [10,10] or [25] -> [10,10,5]
def split_list(input_list: list, slice_size: int = num_sentence_chunk_size) -> list[list[str]]:
    return [input_list[i:i+slice_size] for i in  range (0, len(input_list), slice_size)]

test_list = list(range(25))
split_list(test_list)

[[0, 1, 2, 3, 4, 5, 6, 7, 8, 9],
 [10, 11, 12, 13, 14, 15, 16, 17, 18, 19],
 [20, 21, 22, 23, 24]]

In [74]:
# loop through pages and texts and split sentences into chunks
for item in tqdm(pages_and_texts):
    item["sentence_chunks"] = split_list(input_list = item["sentences"],
                                         slice_size = num_sentence_chunk_size
                                        )
    item["num_chunks"] = len(item["sentence_chunks"])

  0%|          | 0/1208 [00:00<?, ?it/s]

In [75]:
random.sample(pages_and_texts, k=1)

[{'page_number': 187,
  'page_char_count': 1528,
  'page_word_count': 255,
  'page_sentence_count_raw': 1,
  'page_token_count': 382.0,
  'text': 'Seasoning  Foods  Allspice  Lean ground meats, stews, tomatoes, peaches, applesauce,  cranberry sauce, gravies, lean meat  Almond  extract  Puddings, fruits  Caraway  seeds  Lean meats, stews, soups, salads, breads, cabbage,  asparagus, noodles  Chives  Salads, sauces, soups, lean-meat dishes, vegetables  Cider  vinegar  Salads, vegetables, sauces  Cinnamon  Fruits, breads, pie crusts  Curry  powder  Lean meats (especially lamb), veal, chicken, fish, tomatoes,  tomato soup, mayonnaise,  Dill  fish sauces, soups, tomatoes, cabbages, carrots,  cauliflower, green beans, cucumbers, potatoes, salads,  macaroni, lamb  Garlic (not  garlic salt)  Lean meats, fish, soups, salads, vegetables, tomatoes,  potatoes  Ginger  Chicken, fruits  Lemon  juice  Lean meats, fish, poultry, salads, vegetables  Mace  Hot breads, apples, fruit salads, carrots, cauli

In [76]:
df = pd.DataFrame(pages_and_texts)
df.describe().round(2)

,page_number,page_char_count,page_word_count,page_sentence_count_raw,page_token_count,page_sentence_count_spacy,num_chunks
count,1208.00,1208.00,1208.00,1208.00,1208.00,1208.00,1208.00
mean,562.50,1148.00,198.30,9.97,287.00,10.32,1.53
std,348.86,560.38,95.76,6.19,140.10,6.30,0.64
min,-41.00,0.00,1.00,1.00,0.00,0.00,0.00
25%,260.75,762.00,134.00,4.00,190.50,5.00,1.00
50%,562.50,1231.50,214.50,10.00,307.88,10.00,1.00
75%,864.25,1603.50,271.00,14.00,400.88,15.00,2.00
max,1166.00,2308.00,429.00,32.00,577.00,28.00,3.00


### Splitting each chunk into its own item

we'd like to embed each chunck of sentences into it's own numerical representation.

that'll give us a good level of granuality.

Meaning, we can dive specifically into the text sample that was used in our model.

In [77]:
import re

# split each chunk into its own item
pages_and_chunks = []
for item in tqdm(pages_and_texts[:1]):
    print(item)

  0%|          | 0/1 [00:00<?, ?it/s]

{'page_number': -41, 'page_char_count': 29, 'page_word_count': 4, 'page_sentence_count_raw': 1, 'page_token_count': 7.25, 'text': 'Human Nutrition: 2020 Edition', 'sentences': ['Human Nutrition: 2020 Edition'], 'page_sentence_count_spacy': 1, 'sentence_chunks': [['Human Nutrition: 2020 Edition']], 'num_chunks': 1}


In [78]:
import re

# split each chunk into its own item
pages_and_chunks = []
for item in tqdm(pages_and_texts):
    for sentence_chunk in item["sentence_chunks"]:
        chunk_dict = {}
        chunk_dict["page_number"] = item["page_number"]

        # join the sentences together into a pragraph-like struncture
        joined_sentence_chunk = "".join(sentence_chunk).replace(" "," ").strip()
        joined_sentence_chunk = re.sub(r'\.([A-Z])',r'. \1', joined_sentence_chunk)
        chunk_dict["sentence_chunk"] = joined_sentence_chunk

        # get some stats oon our chunks
        chunk_dict["chunk_char_count"] = len(joined_sentence_chunk)
        chunk_dict["chunk_word_count"] = len([word for word in joined_sentence_chunk.split(" ")])
        chunk_dict["chunk_token_count"] = len(joined_sentence_chunk) / 4 # 1 token = ~4 chars
        pages_and_chunks.append(chunk_dict)

len(pages_and_chunks)

  0%|          | 0/1208 [00:00<?, ?it/s]

1843

In [79]:
random.sample(pages_and_chunks, k=1)

[{'page_number': -27,
  'sentence_chunk': 'The Body’s Offense  University of Hawai‘i at Mānoa Food Science and  Human Nutrition Program and Human Nutrition  Program  597  Phytochemicals  University of Hawai‘i at Mānoa Food Science and  Human Nutrition Program and Human Nutrition  Program  600  Part\xa0X.\xa0Chapter 10. Major Minerals  Introduction  University of Hawai‘i at Mānoa Food Science and  Human Nutrition Program and Human Nutrition  Program  609  Calcium  University of Hawai‘i at Mānoa Food Science and  Human Nutrition Program and Human Nutrition  Program  613  Phosphorus  University of Hawai‘i at Mānoa Food Science and  Human Nutrition Program and Human Nutrition  Program  631  Sulfur  University of Hawai‘i at Mānoa Food Science and  Human Nutrition Program and Human Nutrition  Program  635  Magnesium  University of Hawai‘i at Mānoa Food Science and  Human Nutrition Program and Human Nutrition  Program  640',
  'chunk_char_count': 880,
  'chunk_word_count': 160,
  'chunk_token

In [80]:
df = pd.DataFrame(pages_and_chunks)
df.describe().round(2)

,page_number,chunk_char_count,chunk_word_count,chunk_token_count
count,1843.00,1843.00,1843.00,1843.00
mean,583.38,750.01,127.90,187.50
std,347.79,455.70,79.99,113.93
min,-41.00,14.00,4.00,3.50
25%,280.50,321.50,51.00,80.38
50%,586.00,763.00,131.00,190.75
75%,890.00,1137.50,194.00,284.38
max,1166.00,1870.00,412.00,467.50


### Filter chunks with under 30 tokens in Length

These chunks may not contain much useful information

In [81]:
# Show random shunks with under 30 tokens in length
min_token_lnegth = 30
for row in df[df["chunk_token_count"] <= min_token_lnegth].sample(5).iterrows():
    print(f'chunk token count : {row[1]["chunk_token_count"]} | Text : {row[1]["sentence_chunk"]}')

chunk token count : 29.0 | Text : A concentration gradient is a form of potential energy, like water  172  |  Electrolytes Important for Fluid Balance
chunk token count : 20.75 | Text : PART XVIII  CHAPTER 18. NUTRITIONAL ISSUES  Chapter 18. Nutritional Issues  |  1041
chunk token count : 24.75 | Text : http://pressbooks.oer.hawaii.edu/ humannutrition2/?p=485  930  |  Older Adulthood: The Golden Years
chunk token count : 10.25 | Text : PART XIX  APPENDICES  Appendices  |  1149
chunk token count : 4.75 | Text : Introduction  |  61


In [82]:
# Filter out DataFrame for rows with under 30 tokens
pages_and_chunks_over_min_token_len = df[df["chunk_token_count"] > min_token_lnegth].to_dict(orient="records")
pages_and_chunks_over_min_token_len[:2]

[{'page_number': -39,
  'sentence_chunk': 'Human Nutrition: 2020  Edition  UNIVERSITY OF HAWAI‘I AT MĀNOA  FOOD SCIENCE AND HUMAN  NUTRITION PROGRAM  ALAN TITCHENAL, SKYLAR HARA,  NOEMI ARCEO CAACBAY, WILLIAM  MEINKE-LAU, YA-YUN YANG, MARIE  KAINOA FIALKOWSKI REVILLA,  JENNIFER DRAPER, GEMADY  LANGFELDER, CHERYL GIBBY, CHYNA  NICOLE CHUN, AND ALLISON  CALABRESE',
  'chunk_char_count': 320,
  'chunk_word_count': 54,
  'chunk_token_count': 80.0},
 {'page_number': -38,
  'sentence_chunk': 'Human Nutrition: 2020 Edition by University of Hawai‘i at Mānoa Food Science and  Human Nutrition Program is licensed under a Creative Commons Attribution 4.0  International License, except where otherwise noted.',
  'chunk_char_count': 212,
  'chunk_word_count': 32,
  'chunk_token_count': 53.0}]

In [83]:
random.sample(pages_and_chunks_over_min_token_len, k=1)

[{'page_number': 322,
  'sentence_chunk': 'Chylomicron s Contain  Triglycerides  Cholesterol  Molecules  and other  Lipids by  OpenStax  College\xa0/ CC  BY 3.0  Just as lipids require special handling in the digestive tract to move  within a water-based environment, they require similar handling  to travel in the bloodstream. Inside the intestinal cells, the  monoglycerides and fatty acids reassemble themselves into  triglycerides. Triglycerides, cholesterol, and phospholipids form  lipoproteins when joined with a protein carrier. Lipoproteins have  an inner core that is primarily made up of triglycerides and  cholesterol esters (a cholesterol ester is a cholesterol linked to a  fatty acid). The outer envelope is made of phospholipids  interspersed with proteins and cholesterol. Together they form a  chylomicron, which is a large lipoprotein that now enters the  lymphatic system and will soon be released into the bloodstream  via the jugular vein in the neck. Chylomicrons transport fo

### Embedding our text chunks

Embeddings are a broad but powerful concept

while humans understand text, machines understand numbers.

what we'd like to do:
- Turn our text chunks into numbers, specifically embeddings.

A useful numerical representation.

The best part about embedding is that are a *Learned* representaion.

In [84]:
from sentence_transformers import SentenceTransformer
embedding_model = SentenceTransformer(model_name_or_path="all-mpnet-base-v2", device="cpu")

# Create  alist of sentences
sentences = ["Macronutrients  For carbohydrates, the Acceptable Macronutrient Distribution  Range (AMDR) is 45–65 percent of daily calories (which is a  recommended daily allowance of 135–195 grams for 1,200 daily  calories).",
             " Carbohydrates high in fiber should make up the bulk of  intake. The AMDR for protein is 10–30 percent of daily calories  (30–90 grams for 1,200 daily calories).",
             "Children have a high need for  protein to support muscle growth and development."]

# sentences are encoded/embedded by calling model.encode()
embeddings = embedding_model.encode(sentences)
embeddings_dict = dict(zip(sentences, embeddings))

# see the embeddings
for sentence, embedding in embeddings_dict.items():
    print(f"sentence: {sentence}")
    print(f"Embedding: {embedding}")
    print("")

/home/haseebkhan2@ISB.MTBC.COM/.local/lib/python3.10/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


sentence: Macronutrients  For carbohydrates, the Acceptable Macronutrient Distribution  Range (AMDR) is 45–65 percent of daily calories (which is a  recommended daily allowance of 135–195 grams for 1,200 daily  calories).
Embedding: [ 4.78209835e-03 -1.77627839e-02 -1.56176034e-02  1.61853600e-02
  4.53810170e-02 -2.39653164e-03 -7.19852075e-02  2.77123451e-02
 -2.35431157e-02  2.84305331e-03  6.22813851e-02 -3.30506675e-02
 -4.34793830e-02  1.96437202e-02 -7.34178862e-03  1.47870518e-02
 -5.15946280e-03 -3.77205089e-02 -2.53242496e-02  1.11038387e-02
 -2.15416383e-02  5.44781191e-03  4.29924997e-03 -1.46433548e-03
 -6.72092959e-02  9.18062329e-02  8.10700431e-02 -3.34370695e-03
  1.86929405e-02 -4.11523767e-02 -4.16449457e-03  2.39505507e-02
  8.02970864e-03 -4.39422876e-02  1.77624952e-06 -6.40881946e-03
  1.50143690e-02  4.43390831e-02  5.21373563e-03  2.54980437e-02
  4.56635281e-02 -2.57445895e-03  2.82275304e-03  1.29398308e-03
  1.17970901e-02 -4.58622351e-03  5.17865596e-03 -1.

In [ ]:
# %%time

# embedding_model.to("cpu")

# # Embed each chunk one by one
# for item in tqdm(pages_and_chunks_over_min_token_len):
#     item["embedding"] = embedding_model.encode(item["sentence_chunk"])

In [85]:
%%time

embedding_model.to("cuda")

# Embed each chunk one by one
for item in tqdm(pages_and_chunks_over_min_token_len):
    item["embedding"] = embedding_model.encode(item["sentence_chunk"])


  0%|          | 0/1685 [00:00<?, ?it/s]

CPU times: user 2min 9s, sys: 0 ns, total: 2min 9s
Wall time: 52.5 s


In [86]:
%%time

text_chunks = [item["sentence_chunk"] for item in pages_and_chunks_over_min_token_len]
text_chunks[419]

CPU times: user 934 μs, sys: 0 ns, total: 934 μs
Wall time: 959 μs


'Personal Choices  Carbohydrates are in most foods so you have a great variety of  choices with which to meet the carbohydrates recommendations  for a healthy diet. The 2010 Dietary Guidelines recommends eating  more unrefined carbohydrates and more fiber, and reducing  consumption of foods that are high in added sugars. To accomplish  these recommendations use some or all of the following  suggestions:  • Get more daily carbohydrate servings from whole grains by  eating a whole-grain cereal for breakfast, using whole-grain  bread to make a sandwich for lunch, and eating a serving of  beans and/or nuts with dinner. • Make sure to get at least three servings (or more) of all the  grains you eat as whole grains every day. A serving of whole  grains is equal to one slice of whole-wheat bread, one ounce of  whole-grain cereal, and one-half cup of cooked cereal, brown  rice, or whole-wheat pasta. Food products made with cornmeal use the whole grain so  choose tortillas, corn cereals, and co

In [88]:
%%time

# Embed all texts in batches
text_chunk_embeddings = embedding_model.encode(text_chunks, batch_size=10, convert_to_tensor=True)

CPU times: user 27.1 s, sys: 0 ns, total: 27.1 s
Wall time: 13.8 s
